In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
print("✅ CUDA_VISIBLE_DEVICES=0 set")

✅ CUDA_VISIBLE_DEVICES=0 set


In [2]:
# Phase 8 boilerplate: CUDA 13 lib preload + clone latest repo + secrets

# 1. Preload CUDA 13 libraries (fixes bitsandbytes loading on Kaggle)
import ctypes
import glob

for path in glob.glob("/usr/local/lib/python3.12/dist-packages/nvidia/cu13/lib/*.so.13"):
    try:
        ctypes.CDLL(path, mode=ctypes.RTLD_GLOBAL)
    except OSError:
        pass
print("✅ CUDA 13 libraries preloaded")

# 2. Clone the latest repo (for src/data_filter.py and src/train.py)
import subprocess
subprocess.run(["rm", "-rf", "/kaggle/working/repo"], check=False)
subprocess.run(
    ["git", "clone", "https://github.com/arinkc/llm-finetuning-project.git", "/kaggle/working/repo"],
    check=True,
)

import sys
sys.path.insert(0, "/kaggle/working/repo")
print("✅ Repo cloned")

# 3. Suppress harmless warnings
import warnings
warnings.filterwarnings("ignore", category=SyntaxWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
print("✅ Warnings suppressed")

✅ CUDA 13 libraries preloaded


Cloning into '/kaggle/working/repo'...


✅ Repo cloned
✅ Warnings suppressed


In [ ]:
!pip install -q --upgrade \
    "transformers>=4.45.0,<4.50.0" \
    "datasets>=3.0.0" \
    "peft>=0.13.0,<0.15.0" \
    "trl>=0.11.0,<0.13.0" \
    "accelerate>=1.0.0" \
    "wandb>=0.18.0" \
    "sentencepiece" \
    "protobuf"

# Remove torchvision (causes CUDA mismatch — we don't need it for text)
!pip uninstall -y torchvision torchaudio 2>/dev/null

print("✅ Libraries installed — RESTART KERNEL NOW before continuing")

In [3]:
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
hf_token = secrets.get_secret("HF_TOKEN")
wandb_key = secrets.get_secret("WANDB_API_KEY")

from huggingface_hub import login
login(token=hf_token)

import os
os.environ["WANDB_API_KEY"] = wandb_key
os.environ["WANDB_PROJECT"] = "pydoc-llama"

import torch
print(f"✅ Authentication complete")
print(f"   PyTorch: {torch.__version__}")
print(f"   CUDA: {torch.cuda.is_available()}")
print(f"   GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE'}")

✅ Authentication complete
   PyTorch: 2.10.0+cu128
   CUDA: True
   GPU: Tesla T4


In [4]:
import os
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
os.environ["HF_TOKEN"] = secrets.get_secret("HF_TOKEN")
os.environ["WANDB_API_KEY"] = secrets.get_secret("WANDB_API_KEY")
os.environ["WANDB_PROJECT"] = "pydoc-llama"

print(f"✅ HF_TOKEN set ({len(os.environ['HF_TOKEN'])} chars)")
print(f"✅ WANDB_API_KEY set ({len(os.environ['WANDB_API_KEY'])} chars)")

✅ HF_TOKEN set (37 chars)
✅ WANDB_API_KEY set (86 chars)


In [ ]:
# Pull the latest train.py from your repo
!cd /kaggle/working/repo && git pull

# Re-import (Python caches modules; force reload)
import importlib
import src.train
importlib.reload(src.train)
from src.train import TrainingConfig, run_training

# Configure for smoke test: 100 examples, 1 epoch, ~50 steps
cfg = TrainingConfig(smoke_test=True)
print(f"Config: smoke_test={cfg.smoke_test}, epochs={cfg.num_train_epochs}")

# Run it
trainer = run_training(cfg)

In [ ]:
!pip install -q --upgrade bitsandbytes

In [5]:
!cd /kaggle/working/repo && git pull

import importlib
import src.train
importlib.reload(src.train)
from src.train import TrainingConfig, run_training

cfg = TrainingConfig(smoke_test=True)
trainer = run_training(cfg)

Already up to date.


2026-05-13 14:57:49.624589: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778684269.650490    1598 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778684269.658946    1598 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1778684269.692966    1598 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778684269.692985    1598 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778684269.692987    1598 computation_placer.cc:177] computation placer alr

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

...

✅ LoRA configured: 41,943,040 trainable / 4,582,543,360 total (0.9153%)
Loading dataset: Arinkc/pydoc-llama-codesearchnet-curated
⚠️  Smoke test mode — using 100 train / 50 val examples
   Train: 100
   Validation: 50
🚀 Starting training...


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: kcarin123 (kcarin123-salisbury-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Step,Training Loss,Validation Loss


✅ Final adapter saved to /kaggle/working/checkpoints/final


In [6]:
import torch

model = trainer.model
tokenizer = trainer.processing_class

test_function = '''def fibonacci(n):
    if n <= 1:
        return n
    return fibonacci(n-1) + fibonacci(n-2)'''

messages = [
    {"role": "system", "content": "You are an expert Python documentation writer. Given a Python function, generate a concise, Google-style docstring. Output only the docstring text—no surrounding code, no markdown formatting, no preamble."},
    {"role": "user", "content": f"Generate a Google-style docstring for this function:\n\n```python\n{test_function}\n```"},
]

inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    return_tensors="pt",
).to("cuda")

model.eval()
with torch.no_grad():
    outputs = model.generate(
        inputs,
        max_new_tokens=200,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )

response = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
print("=== SMOKE-TEST MODEL OUTPUT ===")
print(response)

/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.6` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:595: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


=== SMOKE-TEST MODEL OUTPUT ===
Compute the nth Fibonacci number.

        Args:
            n (int): The index of the Fibonacci number to compute.

        Returns:
            int: The nth Fibonacci number.

        Raises:
            ValueError: If n is a negative integer.


In [7]:
smoke_test_result = {
    "model": "Llama-3.1-8B-Instruct + LoRA (smoke test, 6 steps)",
    "training_examples": 100,
    "training_steps": 6,
    "prompt": "Generate docstring for fibonacci function",
    "output": """Compute the nth Fibonacci number.

        Args:
            n (int): The index of the Fibonacci number to compute.

        Returns:
            int: The nth Fibonacci number.

        Raises:
            ValueError: If n is a negative integer.""",
    "observations": [
        "Concise imperative-verb summary",
        "Proper Google-style Args/Returns/Raises sections",
        "No markdown formatting or preamble",
        "Clean stop (no postamble)",
    ],
}

import json
with open("/kaggle/working/smoke_test_output.json", "w") as f:
    json.dump(smoke_test_result, f, indent=2)
print("✅ Saved smoke test output")

✅ Saved smoke test output
